# Información y Termodinámica

Este cuaderno ilustra la equivalencia entre entropía de Shannon y entropía de Boltzmann,
el principio de Landauer, y el límite energético de la computación, acompañando el artículo
[Información y Termodinámica](../../05-conexiones-y-aplicaciones/06-informacion-y-termodinamica.md).

Solo se usa la biblioteca estándar de Python.

In [ ]:
import math
import random

## Constantes físicas

In [ ]:
k_B = 1.380649e-23   # constante de Boltzmann (J/K)
T_room = 300.0       # temperatura ambiente (K)
eV = 1.602176634e-19 # electronvoltio (J)

# Límite de Landauer a temperatura ambiente
landauer_J = k_B * T_room * math.log(2)
landauer_eV = landauer_J / eV

print("Constantes fundamentales:")
print(f"  k_B = {k_B:.4e} J/K")
print(f"  T_ambiente = {T_room} K")
print(f"  Límite de Landauer: {landauer_J:.4e} J/bit")
print(f"  Límite de Landauer: {landauer_eV:.6f} eV/bit")
print(f"  Factor conversión bits→termodinámica: k_B·ln2 = {k_B * math.log(2):.4e} J/(K·bit)")

## Ejemplo 1: Entropía de Shannon ↔ Entropía de Boltzmann

Para un sistema con microestados de probabilidades $p_1, \ldots, p_n$:
$$S_{\text{Boltzmann}} = k_B \ln 2 \cdot H_{\text{Shannon}}$$

In [ ]:
def shannon_entropy_bits(probs):
    """Entropía de Shannon en bits."""
    return -sum(p * math.log2(p) for p in probs if p > 0)


def boltzmann_entropy(probs):
    """Entropía de Gibbs-Boltzmann en J/K."""
    return k_B * math.log(2) * shannon_entropy_bits(probs)


# Distintas distribuciones
distribuciones = [
    ([1.0], "1 estado (determinista)"),
    ([0.5, 0.5], "2 estados uniformes"),
    ([0.25, 0.25, 0.25, 0.25], "4 estados uniformes"),
    ([0.5, 0.3, 0.2], "3 estados no uniformes"),
    ([1/6]*6, "dado justo (6 estados)"),
]

print("Equivalencia Shannon ↔ Boltzmann:")
print(f"{'Distribución':>30} | {'H (bits)':>10} | {'S (J/K)':>14} | {'S (J/K)/k_B·ln2':>16}")
print("-" * 80)
for probs, label in distribuciones:
    h = shannon_entropy_bits(probs)
    s = boltzmann_entropy(probs)
    ratio = s / (k_B * math.log(2)) if s > 0 else 0
    print(f"{label:>30} | {h:>10.4f} | {s:>14.4e} | {ratio:>16.4f}")

print("\nNota: S/(k_B·ln2) = H en todos los casos ✓")

## Ejemplo 2: Límite de Landauer — coste de borrar bits

Borrar $n$ bits requiere disipar al menos $n \cdot k_B T \ln 2$ joules.

In [ ]:
def landauer_cost(n_bits, T=T_room):
    """Energía mínima (J) para borrar n_bits a temperatura T."""
    return n_bits * k_B * T * math.log(2)


# Comparación con procesadores reales
# Un smartphone moderno: ~5W, ~10^9 operaciones/s
# Un servidor: ~100W, ~10^12 operaciones/s
# Supercomputador TOP500: ~10MW, ~10^18 FLOPS

print("Límite de Landauer vs. consumo real de procesadores")
print("(a T = 300 K, 1 operación = 1 bit borrado)")
print()

scenarios = [
    ("Smartphone (5W, 10^9 op/s)", 5.0, 1e9),
    ("Servidor (100W, 10^12 op/s)", 100.0, 1e12),
    ("TOP500 #1 (10MW, 10^18 FLOPS)", 1e7, 1e18),
]

for label, power_W, ops_per_s in scenarios:
    energy_per_op_real = power_W / ops_per_s   # J/operación real
    energy_per_op_landauer = landauer_cost(1)   # J/bit (límite)
    ratio = energy_per_op_real / energy_per_op_landauer
    print(f"{label}")
    print(f"  Energía real/op:     {energy_per_op_real:.2e} J")
    print(f"  Límite Landauer/op:  {energy_per_op_landauer:.2e} J")
    print(f"  Factor sobre límite: {ratio:.1e}x")
    print()

## Ejemplo 3: Computación reversible vs. irreversible

Las puertas lógicas irreversibles descartan información → coste de Landauer.
Las puertas reversibles (como Toffoli) preservan toda la información.

In [ ]:
def gate_and(a, b):
    """Puerta AND irreversible: 2 bits entrada → 1 bit salida. Descarta 1 bit."""
    return int(a and b)


def gate_nand(a, b):
    """Puerta NAND irreversible. Descarta 1 bit."""
    return int(not (a and b))


def gate_toffoli(a, b, c):
    """Puerta de Toffoli: (a, b, c) → (a, b, c XOR (a AND b)).
    Es reversible: aplicarla dos veces devuelve el estado original."""
    return (a, b, c ^ (a & b))


# Verificar que Toffoli es su propia inversa
print("Puerta de Toffoli (reversible):")
print(f"{'(a,b,c)':>10} | {'Toffoli(a,b,c)':>16} | {'Toffoli²=original':>20}")
print("-" * 52)
for a in [0, 1]:
    for b in [0, 1]:
        for c in [0, 1]:
            t1 = gate_toffoli(a, b, c)
            t2 = gate_toffoli(*t1)
            ok = '✓' if t2 == (a, b, c) else '✗'
            print(f"{str((a,b,c)):>10} | {str(t1):>16} | {str(t2):>20} {ok}")

# Coste de Landauer de puertas irreversibles
print()
print("Coste de Landauer por puerta (T=300K):")
for gate_name, bits_lost in [('AND', 1), ('NAND', 1), ('OR', 1), ('XOR', 0), ('NOT', 0), ('Toffoli', 0)]:
    cost = landauer_cost(bits_lost)
    print(f"  {gate_name:>10}: descarta {bits_lost} bit(s) → coste mín. {cost:.3e} J")

## Ejemplo 4: KL divergencia y trabajo extraíble

El trabajo máximo extraíble de un sistema en distribución $p$ respecto al equilibrio $q$ es:
$$W_{\max} = k_B T \ln 2 \cdot D_{KL}(p \| q)$$

In [ ]:
def kl_divergence(p, q):
    """D_KL(p || q) en bits."""
    kl = 0.0
    for pi, qi in zip(p, q):
        if pi > 0:
            kl += pi * math.log2(pi / qi)
    return kl


def max_extractable_work(p, q_eq, T=T_room):
    """Trabajo máximo extraíble (J) cuando el equilibrio es q_eq."""
    return k_B * T * math.log(2) * kl_divergence(p, q_eq)


# Sistema de 4 estados, equilibrio = uniforme
n_states = 4
q_eq = [1/n_states] * n_states  # distribución de equilibrio (Boltzmann uniforme)

print("Trabajo máximo extraíble vs. distribución del sistema")
print(f"(n={n_states} estados, equilibrio uniforme, T={T_room}K)")
print()
print(f"{'Distribución p':>30} | {'D_KL (bits)':>12} | {'W_max (J)':>14} | {'W_max (eV)':>12}")
print("-" * 76)

distributions = [
    ([0.25, 0.25, 0.25, 0.25], "uniforme (equilibrio)"),
    ([0.5, 0.25, 0.15, 0.10], "ligeramente sesgada"),
    ([0.7, 0.1, 0.1, 0.1], "muy sesgada"),
    ([1.0, 0.0, 0.0, 0.0], "completamente determinista"),
]

for p, label in distributions:
    if any(pi == 0 and qi > 0 for pi, qi in zip(p, q_eq)):
        continue  # KL infinita
    kl = kl_divergence(p, q_eq)
    w = max_extractable_work(p, q_eq)
    w_ev = w / eV
    print(f"{label:>30} | {kl:>12.5f} | {w:>14.4e} | {w_ev:>12.6f}")

print()
print("Observación: más 'alejada' del equilibrio → más trabajo extraíble.")

## Ejemplo 5: Demonio de Maxwell — ¿cuánta información necesita?

Simulamos N moléculas con velocidades aleatorias. El demonio clasifica rápidas vs. lentas.
Calculamos cuántos bits de información registra el demonio y el coste de borrarlos.

In [ ]:
def simulate_maxwell_demon(N, T=T_room, seed=42):
    """Simula el demonio de Maxwell con N moléculas.
    
    Cada molécula tiene velocidad proporcional a N(0, k_B*T/m).
    El demonio registra: ¿es esta molécula rápida (v > v_media)?
    """
    random.seed(seed)
    
    # Velocidades de Maxwell-Boltzmann (simplificadas como normales)
    velocidades = [abs(random.gauss(0, 1)) for _ in range(N)]
    v_media = sum(velocidades) / N
    
    # El demonio clasifica cada molécula: 1 = rápida, 0 = lenta
    clasificaciones = [1 if v > v_media else 0 for v in velocidades]
    p_rapida = sum(clasificaciones) / N
    p_lenta = 1 - p_rapida
    
    # Información por molécula
    h_por_molecula = -p_rapida * math.log2(p_rapida) - p_lenta * math.log2(p_lenta) if 0 < p_rapida < 1 else 0
    
    # Coste de borrar la memoria del demonio
    bits_totales = h_por_molecula * N  # entropía total de la clasificación
    coste_borrado = landauer_cost(bits_totales, T)
    
    return {
        'N': N,
        'p_rapida': p_rapida,
        'bits_por_molecula': h_por_molecula,
        'bits_totales': bits_totales,
        'coste_borrado_J': coste_borrado,
        'coste_borrado_kT': coste_borrado / (k_B * T),
    }


print("Demonio de Maxwell: información y coste de borrado")
print(f"{'N':>8} | {'p_rápida':>10} | {'bits/mol':>10} | {'bits total':>12} | {'coste (kT)':>12}")
print("-" * 64)
for N in [10, 100, 1000, 10000]:
    r = simulate_maxwell_demon(N)
    print(f"{r['N']:>8} | {r['p_rapida']:>10.4f} | {r['bits_por_molecula']:>10.4f} | {r['bits_totales']:>12.2f} | {r['coste_borrado_kT']:>12.2f}")

print()
print("El coste de borrado (en unidades kT) = bits borrados × ln2 ≈ 0.69 × bits")
print("Esto compensa exactamente la reducción de entropía que el demonio consiguió.")

## Ideas clave

- $S_{\text{Boltzmann}} = k_B \ln 2 \cdot H_{\text{Shannon}}$: la misma cantidad,
  distinto sistema de unidades.
- El **principio de Landauer**: borrar 1 bit requiere disipar $k_B T \ln 2$ de energía;
  este es el coste físico mínimo de la computación irreversible.
- Los procesadores actuales operan $10^4$–$10^8$ veces por encima del límite de Landauer.
- La **puerta de Toffoli** es reversible: aplícala dos veces y recuperas el estado original,
  sin coste de Landauer.
- La **KL divergencia** mide el alejamiento del equilibrio y equivale al trabajo extraíble.

## Ejercicios

1. Calcula el límite de Landauer a T=77K (nitrógeno líquido) y T=4K (helio líquido).
   ¿Cómo cambia respecto a T=300K?
2. Un SSD de 1 TB almacena ~8×10¹² bits. Si se borra completamente a T=300K,
   ¿cuánta energía mínima se disipa? ¿Es medible?
3. Verifica que la puerta CNOT $(a,b) \to (a, a \oplus b)$ es reversible.
   ¿Descarta información?
4. Para la distribución de Boltzmann $p_i \propto e^{-E_i/k_BT}$ con energías
   $E_i = i \cdot \varepsilon$ ($i=0,1,2,3$), calcula $D_{KL}(p \| q)$ donde
   $q$ es la distribución uniforme. ¿Qué significa físicamente?